# COMP64702 RAG Coursework
## Notebook 3: Evaluation

This notebook evaluates generated answers using ROUGE, BLEU,
BERTScore and faithfulness metrics.

In [1]:
# Install dependencies

!pip install rouge-score bert-score nltk

In [2]:
# Imports

import json
from pathlib import Path
import numpy as np

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bertscore_score
import nltk
nltk.download("punkt")

C:\Users\gh300\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gh300\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
# Paths and loading files

DATA_DIR = Path(".")

benchmark_path = DATA_DIR / "benchmark_dataset.json"
outputs_path = DATA_DIR / "test_outputs.json"
eval_path = DATA_DIR / "evaluation_results.json"

with open(benchmark_path, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

with open(outputs_path, "r", encoding="utf-8") as f:
    outputs_payload = json.load(f)

outputs = outputs_payload["results"]

print(f"Loaded {len(benchmark)} benchmark items and {len(outputs)} outputs")

Loaded 13 benchmark items and 13 outputs


In [4]:
# Match benchmark and outputs by query_id

gold_map = {item["query_id"]: item for item in benchmark}
pred_map = {item["query_id"]: item for item in outputs}

common_ids = [qid for qid in gold_map if qid in pred_map]

print(f"Matched {len(common_ids)} query IDs")
print(common_ids)

Matched 13 query IDs
['sa_000', 'sa_001', 'sa_002', 'sa_003', 'sa_004', 'sa_005', 'sa_006', 'sa_007', 'sa_008', 'sa_009', 'sa_010', 'sa_011', 'sa_012']


In [5]:
# Build lists for evaluation

predictions = [pred_map[qid]["response"] for qid in common_ids]
references = [gold_map[qid]["answer"] for qid in common_ids]
retrieved_contexts = [
    [ctx["text"] for ctx in pred_map[qid].get("retrieved_context", [])]
    for qid in common_ids
]

print("Example prediction:\n", predictions[0])
print("\nExample reference:\n", references[0])

Example prediction:
 Biryani is a rice dish made with rice, spices, and usually meat, originating from South Asia.

Example reference:
 Biryani is a mixed rice dish originating in the Indian subcontinent, made with long-grain basmati rice, spices, and meat, vegetables, or eggs. It is closely associated with Mughal cuisine and is particularly famous in the city of Hyderabad as well as in Lucknow, where the dum cooking technique is used to slow-cook the layered rice and meat together.


In [6]:
# Rouge function

rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def rouge_scores(pred, ref):
    scores = rouge.score(ref, pred)
    return {
        "rouge1_f": scores["rouge1"].fmeasure,
        "rouge2_f": scores["rouge2"].fmeasure,
        "rougeL_f": scores["rougeL"].fmeasure,
    }

In [7]:
# Bleu function

smooth = SmoothingFunction().method1

def bleu_score(pred, ref):
    pred_tokens = pred.split()
    ref_tokens = [ref.split()]
    return sentence_bleu(ref_tokens, pred_tokens, smoothing_function=smooth)

In [8]:
# Faithfulness function

def faithfulness_score(prediction, context_texts):
    context = " ".join(context_texts).lower()
    pred_tokens = prediction.lower().split()

    if not pred_tokens:
        return 0.0

    supported = sum(1 for tok in pred_tokens if tok in context)
    return supported / len(pred_tokens)

In [9]:
# Compute BERTScore

P, R, F1 = bertscore_score(predictions, references, lang="en", verbose=True)
bertscore_f = F1.tolist()

Loading weights: 100%|█████████████████████████████████████████████████████████████| 389/389 [00:00<00:00, 1189.74it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:14<00:00, 14.52s/it]


computing greedy matching.


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 28.88it/s]


done in 14.57 seconds, 0.89 sentences/sec


In [10]:
# Evaluate each item

per_item = []

for i, qid in enumerate(common_ids):
    pred = predictions[i]
    ref = references[i]
    ctxs = retrieved_contexts[i]

    rouge_dict = rouge_scores(pred, ref)
    bleu = bleu_score(pred, ref)
    faith = faithfulness_score(pred, ctxs)

    item_result = {
        "query_id": qid,
        "rouge1_f": rouge_dict["rouge1_f"],
        "rouge2_f": rouge_dict["rouge2_f"],
        "rougeL_f": rouge_dict["rougeL_f"],
        "bleu": bleu,
        "bertscore_f": bertscore_f[i],
        "faithfulness": faith,
    }

    per_item.append(item_result)

per_item[:2]

[{'query_id': 'sa_000',
  'rouge1_f': 0.31578947368421056,
  'rouge2_f': 0.16216216216216214,
  'rougeL_f': 0.2894736842105263,
  'bleu': 0.009761059286835667,
  'bertscore_f': 0.9166769981384277,
  'faithfulness': 0.8125},
 {'query_id': 'sa_001',
  'rouge1_f': 0.23529411764705882,
  'rouge2_f': 0.08,
  'rougeL_f': 0.196078431372549,
  'bleu': 0.0013151859480216405,
  'bertscore_f': 0.8842397928237915,
  'faithfulness': 0.3888888888888889}]

In [11]:
# Aggregate metrics

aggregate = {
    "mean_rouge1_f": float(np.mean([x["rouge1_f"] for x in per_item])),
    "mean_rouge2_f": float(np.mean([x["rouge2_f"] for x in per_item])),
    "mean_rougeL_f": float(np.mean([x["rougeL_f"] for x in per_item])),
    "mean_bleu": float(np.mean([x["bleu"] for x in per_item])),
    "mean_bertscore_f": float(np.mean([x["bertscore_f"] for x in per_item])),
    "mean_faithfulness": float(np.mean([x["faithfulness"] for x in per_item])),
}

aggregate

{'mean_rouge1_f': 0.4047384378883516,
 'mean_rouge2_f': 0.17231112002468457,
 'mean_rougeL_f': 0.33112809851533653,
 'mean_bleu': 0.05827848548816697,
 'mean_bertscore_f': 0.9014490017524133,
 'mean_faithfulness': 0.7908236056363076}

## Baseline Comparison

A simple baseline system for this task would use **dense retrieval only**, where documents are retrieved solely based on semantic similarity between query and document embeddings.

Our final system instead uses **hybrid retrieval**, which combines:

- **Dense retrieval** using sentence-transformer embeddings
- **Lexical retrieval** using the BM25 algorithm

Hybrid retrieval improves retrieval performance because it combines:

- **Semantic matching** (capturing meaning and paraphrases)
- **Lexical matching** (capturing exact keyword overlap)

This typically improves recall and robustness, particularly when queries contain important keywords that may not be captured purely through embeddings.

In [12]:
# Save evaluation results

eval_results = {
    "per_item": per_item,
    "aggregate": aggregate
}

with open(eval_path, "w", encoding="utf-8") as f:
    json.dump(eval_results, f, indent=2, ensure_ascii=False)

print(f"Saved evaluation results to {eval_path}")

Saved evaluation results to evaluation_results.json


In [13]:
# Show results clearly

print("Evaluation Results (aggregate):")
for k, v in aggregate.items():
    print(f"{k:30s}: {v:.4f}")

Evaluation Results (aggregate):
mean_rouge1_f                 : 0.4047
mean_rouge2_f                 : 0.1723
mean_rougeL_f                 : 0.3311
mean_bleu                     : 0.0583
mean_bertscore_f              : 0.9014
mean_faithfulness             : 0.7908
